In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip -q install faiss-cpu sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 62.8 MB/s eta 0:00:00:00:0100:01


In [3]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder

from transformers import (
    pipeline,
    AutoTokenizer,
)

from sklearn.metrics.pairwise import cosine_similarity

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print(train.shape)

train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
print("Creating Knowledge Base...")

kb = []

for _, row in train.iterrows():

    correct = row["answer"]

    kb.append(str(row[correct]))

print("Knowledge Base Size:", len(kb))

Creating Knowledge Base...
Knowledge Base Size: 2000


In [6]:
print("Loading SentenceTransformer...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Encoding KB...")

kb_embeddings = embedder.encode(
    kb,
    convert_to_numpy=True,
    show_progress_bar=True,
)

print(kb_embeddings.shape)

Loading SentenceTransformer...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding KB...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 384)


In [7]:
dimension = kb_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(kb_embeddings)

print(index.ntotal)

2000


In [11]:
print("Loading Zero-shot model...")

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1
)

print("Loaded.")

Loading Zero-shot model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded.


In [8]:
cross_encoder = CrossEncoder(

    "cross-encoder/ms-marco-MiniLM-L-6-v2"

)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [9]:
bert_tokenizer = AutoTokenizer.from_pretrained(

    "bert-base-uncased"

)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
OPTIONS = ["A","B","C","D","E"]


def retrieve(prompt,k=5):

    emb = embedder.encode(
        [prompt],
        convert_to_numpy=True,
    )

    distances,indices = index.search(
        emb,
        k,
    )

    return indices[0],distances[0]


def rerank(prompt,docs):

    pairs = [
        [prompt,doc]
        for doc in docs
    ]

    scores = cross_encoder.predict(
        pairs
    )

    order = np.argsort(scores)[::-1]

    return order,scores


def rag_string(context,prompt):

    return f"Context: {context} Question: {prompt}"


def map3(actual,predicted):

    score = 0

    for i,p in enumerate(predicted[:3]):

        if p==actual:

            score=1/(i+1)

    return score

In [12]:
# Q1
row = train.iloc[150]

prompt = str(row["prompt"])

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"]),
]

true_answer = str(row[row["answer"]])

result = zs(
    prompt,
    candidate_labels=labels,
    multi_label=False,
)

score = result["scores"][result["labels"].index(true_answer)]

print("Q1:", round(score,3))

Q1: 0.384


In [22]:
#Q2
row = train.iloc[150]

query = embedder.encode(
    [row["prompt"]],
    convert_to_numpy=True,
)

distances, indices = index.search(query,10)

retrieved = indices[0]

true_doc = kb[150]

rank = None

for i,idx in enumerate(retrieved):

    if kb[idx] == true_doc:

        rank = i+1
        break

print("Retrieved indices")
print(retrieved)

print("Q2 Rank =",rank)


Retrieved indices
[ 663 1701 1269 1532  576  847 1693 1906  168  150]
Q2 Rank = 10


In [14]:
#Q3
docs = [kb[i] for i in retrieved]

pairs = [[row["prompt"],doc] for doc in docs]

scores = cross_encoder.predict(pairs)

order = np.argsort(scores)[::-1]

rank = None

for r,idx in enumerate(order):

    if docs[idx] == true_doc:

        rank = r+1
        break

print("Q3 Rank =",rank)

Q3 Rank = 1


In [15]:
#Q4
row = train.iloc[42]

retrieved,_ = retrieve(row["prompt"],k=5)

docs = [kb[i] for i in retrieved]

context = " ".join(docs)

text = f"Context: {context} Question: {row['prompt']}"

tokens = bert_tokenizer(
    text,
    add_special_tokens=True,
    truncation=False,
)

print("Q4:",len(tokens["input_ids"]))

Q4: 216


In [16]:
#Q5
row = train.iloc[150]

context = kb[150]

rag = f"Context: {context} Question: {row['prompt']}"

labels = [
    row["A"],
    row["B"],
    row["C"],
    row["D"],
    row["E"],
]

result = zs(
    rag,
    candidate_labels=labels,
)

correct = row[row["answer"]]

score = result["scores"][result["labels"].index(correct)]

print("Q5:",round(score,3))

Q5: 0.989


In [17]:
#Q6
row = train.iloc[150]

context = kb[999]

rag = f"Context: {context} Question: {row['prompt']}"

labels = [
    row["A"],
    row["B"],
    row["C"],
    row["D"],
    row["E"],
]

result = zs(
    rag,
    candidate_labels=labels,
)

correct = row[row["answer"]]

score = result["scores"][result["labels"].index(correct)]

print("Q6:",round(score,3))

Q6: 0.529


In [18]:
#Q7
hits = 0

for i in range(100):

    row = train.iloc[i]

    retrieved,_ = retrieve(row["prompt"],k=5)

    docs = [kb[idx] for idx in retrieved]

    truth = row[row["answer"]]

    if truth in docs:

        hits += 1

hit_rate = hits/100*100

print("Hits =",hits)

print("Q7 =",round(hit_rate,1))

Hits = 73
Q7 = 73.0


In [19]:
#Q8
scores=[]

for i in range(20):

    row=train.iloc[i]

    retrieved,_=retrieve(row["prompt"],k=5)

    docs=[kb[idx] for idx in retrieved]

    pairs=[[row["prompt"],doc] for doc in docs]

    ce_scores=cross_encoder.predict(pairs)

    best_doc=docs[np.argmax(ce_scores)]

    rag=f"Context: {best_doc} Question: {row['prompt']}"

    labels=[
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"],
    ]

    result=zs(
        rag,
        candidate_labels=labels,
    )

    ranking=[]

    for label in result["labels"]:

        for letter in OPTIONS:

            if row[letter]==label:

                ranking.append(letter)

    scores.append(
        map3(
            row["answer"],
            ranking,
        )
    )

print("Q8 =",round(np.mean(scores),3))

Q8 = 0.975
